In [48]:
import re, json, yaml
from collections import defaultdict

patterns = dict(
    # Identifiers
    attemptId=r"attempt with (?:the )?id 'INTEGER'",
    categoryId=r"category with id 'INTEGER'",
    chapterId=r"chapter with id 'INTEGER'",
    choiceId=r"choice with id 'INTEGER'",
    commentId=r"comment with id 'INTEGER'",
    courseId=r"course with (?:the )?id 'INTEGER'|curso: INTEGER",
    discussionId=r"discussion with id 'INTEGER'|discussion INTEGER",
    enrolmentId=r"enrolment method '.*?' with id 'INTEGER'",
    eventId=r"event .*? with id 'INTEGER'",
    evidenceId=r"evidence with id 'INTEGER'",
    fieldId=r"field with id 'INTEGER'",
    # forumId, targetForumId (Discussion moved)
    forumId=(r"forum with id 'INTEGER'|forum INTEGER", ["forumId", "targetForumId"]),
    glossaryEntryId=r"glossary entry with id 'INTEGER'",
    gradeId=r"grade with id 'INTEGER'",
    gradeItemId=r"grade item with id 'INTEGER'",
    groupId=r"group with id 'INTEGER'",
    groupingId=r"grouping with id 'INTEGER'",
    # Existe en Mount Orange, no lo veo en 'Componentes y eventos.json'
    #h5pId=r"H5P with the id",
    itemId=r"Item (?:created )?with ID INTEGER|item type '.*?' with id 'INTEGER'",
    moduleId=r"course module (?:with )?id 'INTEGER'",
    noteId=r"note with id 'INTEGER'",
    optionId=r"option with id 'INTEGER'",
    overrideId=r"override with id 'INTEGER'",
    # pageId, prevPageId, nextPageId (Page moved)
    pageId=(r"page with (?:the )?id 'INTEGER'", ["pageId", "prevPageId", "nextPageId"]),
    # Quizá pueda implementarse la captura de strings
    #pageUrl=r"page with URL 'STRING'",
    postId=r"post with id 'INTEGER'",
    questionId=r"question with id 'INTEGER'",
    questionCategoryId=r"question category with id 'INTEGER'",
    recordId=r"data record with id 'INTEGER'",
    roleId=r"role with id 'INTEGER'",
    ruleId=r"rule with id 'INTEGER'",
    scormId=r"sco with id 'INTEGER'",
    scormValue=r"value of 'INTEGER'",
    sectionId=r"section number 'INTEGER'",
    stepId=r"\(id 'INTEGER'\)",
    stepIndex=r"step index 'INTEGER'",
    submissionId=r"submission (?:with id (?:of )?)?'INTEGER'",
    submissionWords=r"with 'INTEGER' words",
    submissionFiles=r"uploaded 'INTEGER'",
    # Puede ser eventId? (Calendar subscription updated)
    # "User INTEGER has updated a calendar subscription with id INTEGER of event type course."
    subscriptionId=r"subscription with id 'INTEGER'",
    tagId=r"tag with id 'INTEGER'",
    tourId=r"tour with id 'INTEGER'",
    userCompetencyId=r"user (?:course )?competency with id",
    userCompetencyRating=r"with 'INTEGER' rating",
    # userId, targetUserId (An extension has been granted.)
    userId=(r"user with (?:the )?id 'INTEGER'|user INTEGER|User INTEGER", ["userId", "targetUserId"]),
)

# Infiere campos dentro del texto de una descripción; Por ejemplo:
# `The user with id '20' subscribed the user with id '20' to the discussion  with id '214' in the forum with the course module id '1169'.`
# Campos en orden de aparición: userId, targetUserId, discussionId, moduleId.
#
# De manera que, extrayendo los números enteros en orden y emparejándolos con sus campos se tiene:
# Campos:  userId targetUserId discussionId moduleId
# Valores: 20     20           214          1169
#
def infer_fields(description):
    matches = []
    for field, config in patterns.items():
        # Si es una tupla se obtiene la regex de la posición 0
        pattern = config[0] if isinstance(config, tuple) else config
        for match in re.finditer(pattern, description, re.IGNORECASE):
            matches.append((match.start(), field))

    # Las tuplas son del tipo: (posición, identificador),
    # por lo que al ordenarlas por posición, valga la redundancia,
    # los campos se guardan en el orden de aparición en la descripción
    matches.sort(key=lambda x: x[0])
    fields = [match[1] for match in matches]

    # Diccionario que autogenera un '0' si la clave no existe
    counts = defaultdict(int)

    # Si un campo aparece más de una vez, se obtiene del diccionario 'repeated'
    final_fields = []
    for field in fields:
        config = patterns[field]
        index = counts[field]

        if isinstance(config, tuple):
            names = config[1]
            final_fields.append(names[index])
        else:
            final_fields.append(field)

        counts[field] += 1

    return final_fields

def transform(node):
    if isinstance(node, dict):
        return {k: transform(v) for k, v in node.items()}
    elif isinstance(node, list):
        unique_tuples = dict.fromkeys(tuple(infer_fields(item)) for item in node)
        return [list(t) for t in unique_tuples]
    return node

with open('../components.json') as f:
    data = json.load(f)

mappings = transform(data)

# Impresión de la configuración para el backend.
#
# flow_style=None
# Se evita el flow_style=True porque hace YAML más verboso, utilizando llaves y comas, asemejándose a JSON;
# Tampoco se descarta por completo (flow_style=False) porque se prefieren las listas inline [...].
#
# sort_keys=True
# No es estrictamente necesario, pero se mantienen los componentes y eventos ordenados
#
# allow_unicode=True
# Tampoco es estrictamente necesario ya que se trabaja sobre strings en language=en,
# además, independientemente del idioma, los identificadores mantienen su posición,
# pero se conserva la compatibilidad con Unicode para evitar imprevistos.
#
# width=float("inf")
# Ancho de línea indefinido para que las líneas largas no incluyan saltos de línea.
#
print(yaml.dump(mappings, default_flow_style=None, sort_keys=True, allow_unicode=True, width=float("inf")))

Activity report:
  Activity report viewed:
  - [userId, courseId]
  Outline report viewed:
  - [userId, targetUserId, courseId]
Assignment:
  A submission has been submitted.:
  - [userId, submissionId, moduleId]
  All the submissions are being downloaded.:
  - [userId, moduleId]
  An extension has been granted.:
  - [userId, targetUserId, moduleId]
  Assignment override created:
  - [userId, overrideId, moduleId, targetUserId]
  Course module instance list viewed:
  - [userId, courseId]
  Course module viewed:
  - [userId, moduleId]
  Grading form viewed:
  - [userId, targetUserId, moduleId]
  Grading table viewed:
  - [userId, moduleId]
  Submission confirmation form viewed.:
  - [userId, moduleId]
  Submission form viewed.:
  - [userId, moduleId]
  - [userId, targetUserId, moduleId]
  Submission viewed.:
  - [userId, targetUserId, moduleId]
  The state of the workflow has been updated.:
  - [userId, targetUserId, moduleId]
  The status of the submission has been viewed.:
  - [userId